# Create files with annotations for ECOD and CATH domains

In [31]:
#import libraries
import pandas as pd
import os

## ECOD 
Modify ecod.latest.domains.txt to get the neccessary info
1. Remove lines that are comments
2. Remove unnecessary columns
 * Column 3: ECOD representative status
 * Column 4: ECOD hierachy identifier
 * Column 6: Chain identifier (note: case-sensitive)
 * Column 8: seq_id number range (based on internal PDB indices)
 * Column 15: Domain assembly status
 * Column 16: Comma-separated value list of non-polymer entities within 4 A of at least one residue of domain

In [32]:
#read in the 'ecod.latest.domains.txt'. This has info about all the domains, but we are using a manual selection of pdbs.
#from ecod.latest.pdb.tar.gz, get the uid's of the pdbs and merge with the domain annotations.

#read in the 'ecod.latest.domains.txt'
project_path ='/Users/karenzhu/Desktop/Projects/MAIN_structure_project/work/SP001_050/SP017/'
ecod_domain_file_path = os.path.join(project_path,"ecod.latest.domains.txt")

ecod_domain_df = pd.read_csv(ecod_domain_file_path, sep='\t', header=None, comment='#', dtype=str,
                      names=['uid', 'ecod_domain_id', 'manual_rep', 't_id', 'pdb', 'chain', 'pdb_range', 'seqid_range', 'unp_acc', 'arch_name', 'x_name', 'h_name', 't_name', 'f_name', 'asm_status', 'ligand'])

ecod_domain_df = ecod_domain_df[['uid', 'ecod_domain_id','pdb', 'chain', 'pdb_range','arch_name', 'x_name', 'h_name', 't_name', 'f_name']]

ecod_domain_df['uid'] = ecod_domain_df['uid'].astype(str)

In [33]:
#read in the list of pdbs that are in ecod.latest.pdb.tar.gz
ecod_manual_uids_path = os.path.join(project_path, 'ecod_manual_pdbs_uid_list.txt')
ecod_manual_uids_df = pd.read_csv(ecod_manual_uids_path, sep='\s+', header=None, names= ['uid'], dtype=str)

In [34]:
#get only the annotations for domains in ecod.latest.pdb.tar.gz
ecod_domain_annotations_df = ecod_manual_uids_df.merge(ecod_domain_df, on='uid', how='left')

In [35]:
#save annotations to csv
ecod_domain_annotations_path = os.path.join(project_path, "ECOD_ManualRep_domain_annotations.csv")
ecod_domain_annotations_df.to_csv(ecod_domain_annotations_path, index = False )

## CATH

In [36]:
#file paths
S20_domain_list_path = os.path.join(project_path, 'cath-dataset-nonredundant-S20.list')
cath_domain_list_path = os.path.join(project_path, 'cath-domain-list.txt')
cath_names_path = os.path.join(project_path, 'cath-names.txt')


In [37]:
#read in the data into a pandas df
S20_domain_list_df = pd.read_csv(S20_domain_list_path, sep='\s+', header=None, names= ['Domain_name'])

#S20_domain_list_df.columns = ['Domain_name']
domain_classification_df = pd.read_csv(cath_domain_list_path, sep='\s+', comment='#',header=None, dtype=str,
                                       names=['Domain_name', 'Class', 'Architecture', 'Topology', 'Homologous_superfamily',
                        'S35', 'S60', 'S95', 'S100', 'S100_count', 'Domain_length', 'Structure_resolution'])


In [38]:
#read `cath_names.txt`. Due to the formatting, I can use pandas and will need to parse it 

with open(cath_names_path, 'r') as cath_name_file:
    cath_name_lines = cath_name_file.readlines()[16:]

#no need to parse domain_name since it's representative
cath_node_num = []
cath_node_desc = []
count = 0
for line in cath_name_lines:
    parts = line.split(maxsplit=2)
    cath_node_num.append(parts[0].strip())
    cath_node_desc.append(parts[2].lstrip(':').strip())

#create a dict of the parsed info
cath_names_dict =  dict(zip(cath_node_num,cath_node_desc ))

In [39]:
#create new columns for cath annotations
domain_classification_df['PDB'] = domain_classification_df['Domain_name'].str[:4]

domain_classification_df['CATH_code'] = domain_classification_df['Class'].astype(str) + '.' + \
                              domain_classification_df['Architecture'].astype(str) + '.' + \
                              domain_classification_df['Topology'].astype(str) + '.' + \
                              domain_classification_df['Homologous_superfamily'].astype(str)

domain_classification_df['CATHSOLID'] = domain_classification_df['Class'].astype(str) + '.' + \
                              domain_classification_df['Architecture'].astype(str) + '.' + \
                              domain_classification_df['Topology'].astype(str) + '.' + \
                              domain_classification_df['Homologous_superfamily'].astype(str) + \
                              domain_classification_df['S35'].astype(str) + '.' + \
                              domain_classification_df['S60'].astype(str) + '.' + \
                              domain_classification_df['S95'].astype(str) + '.' + \
                              domain_classification_df['S100'].astype(str) 

#create columns for class, arch, top, and homol to be mapped to cath_names_dict
domain_classification_df['CATH_class'] = domain_classification_df['Class'].astype(str)
domain_classification_df['CATH_architecture'] = domain_classification_df['Class'].astype(str) + '.' + domain_classification_df['Architecture'].astype(str)
domain_classification_df['CATH_topology'] = domain_classification_df['Class'].astype(str) + '.' + domain_classification_df['Architecture'].astype(str) + \
                                            '.' + domain_classification_df['Topology'] .astype(str)
domain_classification_df['CATH_homologous_fam'] = domain_classification_df['CATH_code'].astype(str)

In [40]:
#mapped to cath_names_dict to get annotations
domain_classification_df['CATH_class'] = domain_classification_df['CATH_class'].map(cath_names_dict)

domain_classification_df['CATH_architecture'] = domain_classification_df['CATH_architecture'].map(cath_names_dict)

domain_classification_df['CATH_topology'] = domain_classification_df['CATH_topology'].map(cath_names_dict)

domain_classification_df['CATH_homologous_fam'] = domain_classification_df['CATH_homologous_fam'].map(cath_names_dict)

In [41]:
#extract the chain. 5th character of domain name - chain Character. chain characters of zero ('0') indicate that the PDB file has no chain field.
domain_classification_df['Chain'] = domain_classification_df["Domain_name"].str[4]

In [42]:
#select relevant annotated columns
domain_classification_df = domain_classification_df[['Domain_name', 'PDB', 'Chain','CATH_code','CATHSOLID','Domain_length', 'CATH_class', 'CATH_architecture','CATH_topology','CATH_homologous_fam']]

In [43]:
#get only the annotations for domains in S20 nonredundant set
S20_annotations_df = S20_domain_list_df.merge(domain_classification_df, on='Domain_name', how='left')

In [44]:
#save final CATH annotations to csv file
S20_annotations_path = os.path.join(project_path, "CATH_S20_domain_annotations.csv")
S20_annotations_df.to_csv(S20_annotations_path, index = False)